# Data Transformation
LSE_ID = 250094127

## Loading the json files

First, I **import** the relevant libraries and load the raw movie details. 

In [84]:
import pandas as pd 
import json
import sqlite3

In [85]:
with open("../data/raw/movie_details.json") as f:
    raw_details = json.load(f)

In [86]:
raw_details

{'movies': [{'adult': False,
   'backdrop_path': '/k6EOrckWFuz7I4z4wiRwz8zsj4H.jpg',
   'belongs_to_collection': {'id': 10,
    'name': 'Star Wars Collection',
    'poster_path': '/pWVLFh4OuejTpUaDQbB1C4zoS2p.jpg',
    'backdrop_path': '/iY2ujEY2m68OTTlPFTiHub9joHS.jpg'},
   'budget': 245000000,
   'genres': [{'id': 12, 'name': 'Adventure'},
    {'id': 28, 'name': 'Action'},
    {'id': 878, 'name': 'Science Fiction'}],
   'homepage': 'http://www.starwars.com/films/star-wars-episode-vii',
   'id': 140607,
   'imdb_id': 'tt2488496',
   'origin_country': ['US'],
   'original_language': 'en',
   'original_title': 'Star Wars: The Force Awakens',
   'overview': 'Thirty years after defeating the Galactic Empire, Han Solo and his allies face a new threat from the evil Kylo Ren and his army of Stormtroopers.',
   'popularity': 21.4627,
   'poster_path': '/wqnLdwVXoBjKibFRR5U3y0aDUhs.jpg',
   'production_companies': [{'id': 1,
     'logo_path': '/tlVSws0RvvtPBwViUyOFAO0vcQS.png',
     'name': 'L

## Transforming the data 

After loading the json files, I use json normalize, and set record_path equal to movies so that each movie represents a single row. I also add a year column to the dataframe. 


In [87]:
df_movie_details = pd.json_normalize(raw_details, record_path="movies")

In [88]:
df_movie_details.head(5)

,adult,backdrop_path,budget,genres,homepage,id,imdb_id,origin_country,original_language,original_title,...,tagline,title,video,vote_average,vote_count,belongs_to_collection.id,belongs_to_collection.name,belongs_to_collection.poster_path,belongs_to_collection.backdrop_path,belongs_to_collection
0,False,/k6EOrckWFuz7I4z4wiRwz8zsj4H.jpg,245000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.starwars.com/films/star-wars-episod...,140607,tt2488496,[US],en,Star Wars: The Force Awakens,...,Every generation has a story.,Star Wars: The Force Awakens,False,7.248,20688,10.0,Star Wars Collection,/pWVLFh4OuejTpUaDQbB1C4zoS2p.jpg,/iY2ujEY2m68OTTlPFTiHub9joHS.jpg,NaN
1,False,/dF6FjTZzRTENfB4R17HDN20jLT2.jpg,150000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 878, ...",https://www.jurassicworld.com/,135397,tt0369610,[US],en,Jurassic World,...,The park is open.,Jurassic World,False,6.703,21726,328.0,Jurassic Park Collection,/qIm2nHXLpBBdMxi8dvfrnDkBUDh.jpg,/njFixYzIxX8jsn6KMSEtAzi4avi.jpg,NaN
2,False,/ehzI1mVcnHqB58NqPyQwpMqcVoz.jpg,190000000,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",https://www.universalpicturesathome.com/movies...,168259,tt2820852,[US],en,Furious 7,...,Vengeance hits home.,Furious 7,False,7.216,11381,9485.0,The Fast and the Furious Collection,/zOCnMPoUxgJK1RFPfN4PcnT16gr.jpg,/z5A5W3WYJc3UVEWljSGwdjDgQ0j.jpg,NaN
3,False,/kIBK5SKwgqIIuRKhhWrJn3XkbPq.jpg,235000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",https://www.marvel.com/movies/avengers-age-of-...,99861,tt2395427,[US],en,Avengers: Age of Ultron,...,A new age has come.,Avengers: Age of Ultron,False,7.275,24664,86311.0,The Avengers Collection,/yFSIUVTCvgYrpalUktulvk3Gi5Y.jpg,/2UNUv4NJdC36E5myDHACBJ99EwL.jpg,NaN
4,False,/wKrxeY6lbu7KFBsWVcMH6M8avwr.jpg,74000000,"[{'id': 10751, 'name': 'Family'}, {'id': 16, '...",http://www.minionsmovie.com/,211672,tt2293640,[US],en,Minions,...,Go back to where it all began.,Minions,False,6.421,11184,544669.0,Minions Collection,/lqU48HkuPDpbumwHk9syT7FbxpC.jpg,/62Qe28oi9PaK3P2ljDYUDTGAyST.jpg,NaN


I only pick the relevant columns for my analysis and the **export** it to a csv file

In [89]:
cols_to_keep = ['id', 'title', 'original_title', 'release_date',
                 'budget', 'revenue', 'popularity', 'vote_average', 
                 'vote_count', 'runtime', 'original_language']

df_movie_details = df_movie_details[cols_to_keep]

In [90]:
df_movie_details["year"] = pd.to_datetime(df_movie_details["release_date"]).dt.year

I figured that TMDB probably uses `runtime = 0` as a placeholder when it has no runtime data for a film, rather than the film genuinely being zero minutes long. Since runtime is central to this analysis, I drop those rows here so I don't have to keep filtering them out downstream. Note this only catches exact zeros — a handful of films still have implausibly short but nonzero runtimes (as low as 1 minute), which slip past this filter and are instead flagged later, in NB03.

In [91]:
df_movie_details = df_movie_details[df_movie_details["runtime"] != 0]

In [92]:
df_movie_details.to_csv('../data/processed/movies.csv')

In [93]:
df_movie_details

,id,title,original_title,release_date,budget,revenue,popularity,vote_average,vote_count,runtime,original_language,year
0,140607,Star Wars: The Force Awakens,Star Wars: The Force Awakens,2015-12-15,245000000,2068223624,21.4627,7.248,20688,136,en,2015
1,135397,Jurassic World,Jurassic World,2015-06-06,150000000,1671537444,18.7528,6.703,21726,124,en,2015
2,168259,Furious 7,Furious 7,2015-04-01,190000000,1515400000,13.9605,7.216,11381,138,en,2015
3,99861,Avengers: Age of Ultron,Avengers: Age of Ultron,2015-04-22,235000000,1405403694,33.7251,7.275,24664,141,en,2015
4,211672,Minions,Minions,2015-06-17,74000000,1159457503,10.9013,6.421,11184,91,en,2015
...,...,...,...,...,...,...,...,...,...,...,...,...
1095,1230368,Springsteen: Deliver Me from Nowhere,Springsteen: Deliver Me from Nowhere,2025-10-22,55000000,45240624,4.2320,6.631,321,120,en,2025
1096,1539104,JUJUTSU KAISEN: Execution,劇場版 呪術廻戦「渋谷事変 特別編集版」×「死滅回游 先行上映」,2025-11-07,0,44457030,18.0459,6.041,109,88,ja,2025
1097,1408208,Exit 8,８番出口,2025-08-29,1400000,43892680,13.3526,7.004,643,95,ja,2025
1098,1233575,Black Bag,Black Bag,2025-03-12,50000000,43887905,6.7802,6.367,1294,94,en,2025


Since I'll be grouping data according to movie genres after, I make a dataframe in which each row represents one single genre. I drop the duplicates after, so as to not have repeated genres. I then export it into a csv. 

In [94]:
df_genres = pd.json_normalize(
    raw_details['movies'],
    record_path=['genres'],
    record_prefix='genre_' 
)

df_genres

,genre_id,genre_name
0,12,Adventure
1,28,Action
2,878,Science Fiction
3,12,Adventure
4,878,Science Fiction
...,...,...
3187,9648,Mystery
3188,53,Thriller
3189,878,Science Fiction
3190,53,Thriller


In [95]:
df_genres = df_genres.drop_duplicates()

In [96]:
df_genres.to_csv("../data/processed/genres.csv")

Furthermore, I also create a dataframe that connects both the movies and the genres. This dataframe will hold information about the genre_id, as well as the movie id. After, I export it into a csv file. 

I filter this table down to only the movie IDs that survived the `runtime != 0` cleanup above. Without this, `df_genres_movie` — built directly from the raw JSON — would still include genre links for movies I've already dropped, which would violate the foreign key from `movie_genres` to `movies` once the schema below enforces it.

In [97]:
df_genres_movie = pd.json_normalize(
    raw_details['movies'],
    record_path=['genres'],
    meta=['id'],
    record_prefix='genre_' 
)

df_genres_movie = df_genres_movie[df_genres_movie["id"].isin(df_movie_details["id"])]

In [98]:
df_genres_movie.to_csv("../data/processed/genres_movie.csv")

## Making a database

First, I make a connection to the database and enforce database integrity

In [99]:
conn = sqlite3.connect("../data/movies.db")

In [100]:
conn.execute("PRAGMA foreign_keys = ON;")

This block of code is written in SQL, and it creates the tables.

I split the data into **three tables** instead of one wide table: `movies` holds one row per film; `genres` holds each unique genre once, keyed by `genre_id`; and `movie_genres` is a join table linking movie IDs to genre IDs. A join table is necessary because the relationship is many-to-many — one film can have several genres, and one genre covers many films — which a single flat table can't represent without duplicating movie rows once per genre. The `FOREIGN KEY` on `movie_genres.id` ties every genre link back to a real row in `movies`, which is what `PRAGMA foreign_keys = ON` above actually enforces.

In [101]:
schema_query_movies = """
DROP TABLE IF EXISTS movies;

CREATE TABLE movies (
    id BIGINT PRIMARY KEY,
    title VARCHAR(255),
    original_title VARCHAR(255),
    release_date DATE,
    budget BIGINT,
    revenue BIGINT,
    popularity DECIMAL(10,4),
    vote_average DECIMAL(4,3),
    vote_count INTEGER,
    runtime INTEGER,
    original_language VARCHAR(10),
    year INT
);
"""
schema_query_genres = """

DROP TABLE IF EXISTS genres;

CREATE TABLE genres (
    genre_id INTEGER PRIMARY KEY,
    genre_name VARCHAR(50) NOT NULL
)


"""

schema_query_genres_movies = """
DROP TABLE IF EXISTS movie_genres;


CREATE TABLE movie_genres (
    id BIGINT,
    genre_id INTEGER,
    genre_name VARCHAR(50),
    PRIMARY KEY (id, genre_id),
    FOREIGN KEY (id) REFERENCES movies(id)
);
"""

conn.executescript(schema_query_genres)
conn.executescript(schema_query_genres_movies)
conn.executescript(schema_query_movies)

After that, pandas is able to read the csv files and populates each table with the correct dataframe.

In [102]:
df_movie_details.to_sql(
    "movies",
    conn,
    if_exists="append", 
    index=False
)

df_genres.to_sql(
    "genres",
    conn,
    if_exists="append", 
    index=False

)


df_genres_movie.to_sql(
    "movie_genres",
    conn,
    if_exists="append", 
    index=False
)


3175

This query shows the tables created in the database

In [103]:
tables_df = pd.read_sql("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

print(tables_df)


           name
0        genres
1  movie_genres
2        movies


## Summary

This notebook turned the raw TMDB JSON into three normalized tables — `movies`, `genres`, and `movie_genres` — written both to CSVs in `data/processed/` and to a SQLite database at `data/movies.db`. NB03 reads from this database to build the analysis.